# Bài 2: Bài Tập Thực Hành - Tối Ưu Hóa Tham Số LightGBM

**Hướng dẫn:**
- Hoàn thành các câu hỏi và bài tập dưới đây để củng cố kiến thức về các tham số của LightGBM.
- Cố gắng tự trả lời trước khi xem đáp án.
- Chạy các ô code để kiểm tra kết quả của bạn.

---

### Phần 1: Câu hỏi lý thuyết

**Câu 1:** Tham số `num_leaves` và `max_depth` đều dùng để kiểm soát độ phức tạp của cây. Trong LightGBM, tại sao `num_leaves` thường được coi là tham số quan trọng hơn để tinh chỉnh? Tăng `num_leaves` ảnh hưởng đến mô hình như thế nào?

**Câu 2:** Giải thích mối quan hệ giữa `learning_rate` và `n_estimators`. Nếu bạn giảm `learning_rate` từ 0.1 xuống 0.01, bạn nên điều chỉnh `n_estimators` như thế nào và tại sao?

**Câu 3:** Overfitting là một vấn đề lớn khi xây dựng mô hình. Liệt kê ít nhất 3 cặp tham số trong LightGBM mà bạn có thể sử dụng để giúp mô hình chống lại overfitting và giải thích ngắn gọn cách chúng hoạt động.

**Câu 4:** Tham số `feature_fraction` (hoặc `colsample_bytree`) và `bagging_fraction` (hoặc `subsample`) có tác dụng gì? Khi nào bạn nên xem xét sử dụng chúng?

**Câu 5:** Early stopping là một kỹ thuật rất hữu ích. Nó giúp giải quyết vấn đề gì khi huấn luyện mô hình Gradient Boosting? Cần những gì để sử dụng early stopping trong LightGBM?

---

### Phần 2: Bài tập thực hành

**Bối cảnh:** Chúng ta tiếp tục với bộ dữ liệu dự đoán bệnh tim (`heart.csv`). Lần này, bạn sẽ thử nghiệm các bộ tham số khác nhau để cải thiện hiệu suất mô hình.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score

# Tải và chuẩn bị dữ liệu
url = 'https://raw.githubusercontent.com/justmarkham/DAT8/master/data/heart.csv'
df_heart = pd.read_csv(url)
X = df_heart.drop('target', axis=1)
y = df_heart['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

**Bài tập 6: Xây dựng mô hình Baseline**

1.  Huấn luyện một mô hình `lgb.LGBMClassifier` với các tham số mặc định (`random_state=42`).
2.  Đánh giá mô hình trên tập test bằng cả `accuracy` và `roc_auc_score`.

In [ ]:
# Viết code của bạn vào đây
clf_base = ...
clf_base.fit(...)

y_pred_base = clf_base.predict(X_test)
y_proba_base = clf_base.predict_proba(X_test)[:, 1]

print("--- Baseline Model ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_base):.4f}")

**Bài tập 7: Xây dựng mô hình phức tạp (dễ overfitting)**

1.  Tạo một mô hình `lgb.LGBMClassifier` với các tham số sau để cố tình làm nó phức tạp hơn:
    - `n_estimators=500`
    - `learning_rate=0.1`
    - `num_leaves=100`
    - `random_state=42`
2.  Huấn luyện và đánh giá tương tự như trên.

In [ ]:
# Viết code của bạn vào đây
clf_complex = ...
clf_complex.fit(...)

y_pred_complex = clf_complex.predict(X_test)
y_proba_complex = clf_complex.predict_proba(X_test)[:, 1]

print("\n--- Complex Model ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_complex):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_complex):.4f}")

**Bài tập 8: Xây dựng mô hình được điều chuẩn (Regularized)**

1.  Tạo một mô hình `lgb.LGBMClassifier` với các tham số nhằm chống overfitting:
    - `n_estimators=1000` (một số lớn để early stopping có thể hoạt động)
    - `learning_rate=0.02`
    - `num_leaves=15`
    - `colsample_bytree=0.8`
    - `subsample=0.8`
    - `reg_alpha=0.1`
    - `reg_lambda=0.1`
    - `random_state=42`
2.  Huấn luyện mô hình, nhưng lần này hãy sử dụng **early stopping**. Dùng `eval_set` là tập test và dừng lại nếu `roc_auc` không cải thiện sau 50 vòng.
3.  Đánh giá mô hình và in ra số vòng lặp tốt nhất (`best_iteration_`).

In [ ]:
# Viết code của bạn vào đây
clf_regularized = ...

clf_regularized.fit(X_train, y_train,
                    eval_set=[(X_test, y_test)],
                    eval_metric='auc',
                    callbacks=[lgb.early_stopping(50, verbose=False)])

y_pred_reg = clf_regularized.predict(X_test)
y_proba_reg = clf_regularized.predict_proba(X_test)[:, 1]

print("\n--- Regularized Model ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_reg):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_reg):.4f}")
print(f"Best iteration: {clf_regularized.best_iteration_}")

---

## Đáp Án

### Phần 1: Lý thuyết

**Câu 1:**
Trong LightGBM, `num_leaves` quan trọng hơn vì nó kiểm soát trực tiếp số lượng lá cuối cùng của cây, điều này quyết định độ phức tạp của mô hình. LightGBM phát triển cây theo kiểu `leaf-wise`, tức là nó luôn chọn lá có khả năng giảm loss nhiều nhất để phân chia. Do đó, việc giới hạn `num_leaves` là cách trực tiếp nhất để kiểm soát sự phát triển này. Tăng `num_leaves` cho phép mô hình tạo ra các cây phức tạp hơn, có khả năng học các mối quan hệ tinh vi trong dữ liệu, nhưng đồng thời làm tăng đáng kể nguy cơ overfitting.

**Câu 2:**
`learning_rate` (tốc độ học) và `n_estimators` (số lượng cây) có mối quan hệ nghịch đảo. `learning_rate` xác định mức độ đóng góp của mỗi cây vào mô hình tổng thể. 
Nếu bạn giảm `learning_rate` (ví dụ: từ 0.1 xuống 0.01), mỗi cây sẽ đóng góp ít hơn vào việc sửa lỗi. Do đó, bạn cần **tăng `n_estimators`** (thường là tăng gấp 10 lần tương ứng) để cho mô hình có đủ "bước" nhỏ để hội tụ đến điểm tối ưu. Chiến lược này (tốc độ học nhỏ, nhiều cây) thường tạo ra các mô hình có khả năng tổng quát hóa tốt hơn và ít bị overfitting hơn.

**Câu 3:** Ba cặp tham số để chống overfitting:
1.  **`num_leaves` (giảm) và `max_depth` (giảm):** Giảm các giá trị này sẽ làm cho cây nông và đơn giản hơn, ngăn chúng học thuộc lòng các chi tiết nhiễu trong tập huấn luyện.
2.  **`min_child_samples` (tăng) và `min_split_gain` (tăng):** Tăng các giá trị này làm cho thuật toán "khó tính" hơn khi quyết định một phép chia. Nó đòi hỏi một nhánh phải có đủ số lượng mẫu hoặc một phép chia phải mang lại lợi ích đủ lớn, từ đó tránh tạo ra các nhánh chỉ học trên vài điểm dữ liệu.
3.  **`feature_fraction` (giảm) và `bagging_fraction` (giảm):** Các tham số này đưa yếu tố ngẫu nhiên vào quá trình huấn luyện. `feature_fraction` chọn một tập con các đặc trưng cho mỗi cây, trong khi `bagging_fraction` chọn một tập con các mẫu dữ liệu. Điều này làm cho các cây trong chuỗi boosting trở nên đa dạng hơn và ít phụ thuộc vào các đặc trưng hoặc mẫu dữ liệu cụ thể, giúp giảm phương sai và chống overfitting.

**Câu 4:**
- **`feature_fraction`:** Ở mỗi cây, chỉ chọn một phần ngẫu nhiên các đặc trưng để xem xét khi tìm điểm chia tốt nhất. Điều này giúp tăng tốc độ (vì không phải duyệt qua tất cả các đặc trưng) và chống overfitting (vì các cây sẽ không quá phụ thuộc vào một vài đặc trưng mạnh).
- **`bagging_fraction`:** Ở mỗi vòng lặp, chỉ sử dụng một phần ngẫu nhiên của dữ liệu để huấn luyện cây. Điều này cũng giúp chống overfitting và làm cho mô hình ổn định hơn.
Bạn nên xem xét sử dụng chúng khi: (1) có số lượng lớn đặc trưng hoặc mẫu dữ liệu, (2) mô hình có dấu hiệu overfitting, hoặc (3) muốn tăng tốc độ huấn luyện.

**Câu 5:**
Early stopping giúp giải quyết vấn đề **xác định số lượng `n_estimators` tối ưu**. Thay vì phải đoán một con số, early stopping sẽ theo dõi hiệu suất của mô hình trên một tập dữ liệu xác thực (validation set). Nếu hiệu suất không cải thiện sau một số vòng lặp nhất định (`stopping_rounds`), quá trình huấn luyện sẽ tự động dừng lại. Điều này giúp tìm ra điểm cân bằng, nơi mô hình đủ phức tạp để học tốt nhưng chưa bắt đầu overfitting.
Để sử dụng early stopping, bạn cần cung cấp:
1.  Một tập dữ liệu xác thực (`eval_set`).
2.  Một metric để theo dõi (`eval_metric`).
3.  Số vòng lặp để kiên nhẫn chờ (`stopping_rounds`) thông qua `lgb.early_stopping` callback.

### Phần 2: Thực hành (Code đáp án)

In [ ]:
# Bài tập 6: Xây dựng mô hình Baseline
clf_base = lgb.LGBMClassifier(random_state=42)
clf_base.fit(X_train, y_train)

y_pred_base = clf_base.predict(X_test)
y_proba_base = clf_base.predict_proba(X_test)[:, 1]

print("--- Baseline Model ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_base):.4f}")

In [ ]:
# Bài tập 7: Xây dựng mô hình phức tạp (dễ overfitting)
clf_complex = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.1,
    num_leaves=100,
    random_state=42
)
clf_complex.fit(X_train, y_train)

y_pred_complex = clf_complex.predict(X_test)
y_proba_complex = clf_complex.predict_proba(X_test)[:, 1]

print("\n--- Complex Model ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_complex):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_complex):.4f}")

In [ ]:
# Bài tập 8: Xây dựng mô hình được điều chuẩn (Regularized)
clf_regularized = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=15,
    colsample_bytree=0.8,
    subsample=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42
)

clf_regularized.fit(X_train, y_train,
                    eval_set=[(X_test, y_test)],
                    eval_metric='auc',
                    callbacks=[lgb.early_stopping(50, verbose=False)])

y_pred_reg = clf_regularized.predict(X_test)
y_proba_reg = clf_regularized.predict_proba(X_test)[:, 1]

print("\n--- Regularized Model ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_reg):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_reg):.4f}")
print(f"Best iteration: {clf_regularized.best_iteration_}")